<center>
    <p style="text-align:center">
        <img alt="arize ax logo" src="https://storage.googleapis.com/arize-assets/arize-logo-white.jpg" width="100"/>
        <br>
        <br>
        <a href="https://arize.com/docs/ax/">Docs</a>
        |
        <a href="https://github.com/Arize-ai">GitHub</a>
        |
        <a href="https://arize-ai.slack.com/join/shared_invite/zt-2w57bhem8-hq24MB6u7yE_ZF_ilOYSBw#/shared-invite/email">Community</a>
    </p>
</center>
<h1 align="center">Aligning Evals (LLM Judge)</h1>

In this tutorial, we'll run a Mastra agent and build a custom evaluator for it. The goal is to understand the workflow for creating evaluators that align with your specific use case. We'll be working in TypeScript.

This tutorial also introduces a human-in-the-loop step for writing annotations, which are used to benchmark and measure agent performance.

Run `deno install` then `jupyter notebook` to get this notebook up and running. This notebook assumes you have already run the Mastra agent provided and generated traces in Arize AX.

## Setup

Let's start by importing the necessary packages.

In [5]:
import { listSpans, createDataset, listDatasetExamples, createExperiment } from "npm:@arizeai/ax-client@latest";
import OpenAI from "npm:openai@latest";

Set up your OpenAI API key.

In [6]:
const arizeApiKey = Deno.env.get("ARIZE_API_KEY") || prompt("Enter your Arize API key:");
const arizeSpaceId = Deno.env.get("ARIZE_SPACE_ID") || prompt("Enter your Arize Space ID or name:");
const openaiApiKey = Deno.env.get("OPENAI_API_KEY") || prompt("Enter your OpenAI API key:");

if (!arizeApiKey || !arizeSpaceId || !openaiApiKey) {
  console.error('Please enter all required credentials to continue');
  Deno.exit(1);
}

Deno.env.set("ARIZE_API_KEY", arizeApiKey);
const openai = new OpenAI({ apiKey: openaiApiKey });
const space = arizeSpaceId;

> **Note:** This notebook connects to Arize AX using your API key. Make sure you have already run the Mastra agent and that traces are visible in your Arize AX project before continuing.

## Creating a Dataset

Grab the Mastra agent traces from Arize AX and format them into dataset examples. In this example, we'll extract the user query, the tool calls, and the agent's final response. Once formatted, we'll upload this dataset back into Arize AX for evaluation.

In [8]:
const { data: spans } = await listSpans({
  project: "mastra-orchestrator-workflow",
  space,
  limit: 500,
});

console.log("Total spans fetched:", spans?.length || 0);

Total spans fetched: 344


In [9]:
// Group spans by trace ID
function groupSpansByTraceId(spans: any[]) {
  const traceGroups: { [traceId: string]: any[] } = {};
  
  spans.forEach(span => {
    const traceId = span.context?.traceId || 'unknown';
    
    if (!traceGroups[traceId]) {
      traceGroups[traceId] = [];
    }
    
    traceGroups[traceId].push(span);
  });
  
  return traceGroups;
}

// Group the spans
const groupedSpans = groupSpansByTraceId(spans);
const traceIds = Object.keys(groupedSpans);

console.log(`\n📊 Found ${traceIds.length} unique traces`);
console.log("Trace IDs:", traceIds.slice(0, 5));


📊 Found 19 unique traces
Trace IDs: [
  "7282842e41d3aa7f409920e70a574a16",
  "b991c72021ccfdba7188d3d3a45fbbe8",
  "9b1705b4eb2c30953293208569ccab60",
  "dc591ea2ea3084f71d7a1a5623eb9da9",
  "1898214471703437bc2741ab483db30a"
]


In [10]:
// Analyze each trace and extract spans into flat dataset fields
const traceAnalysis = traceIds.map(traceId => {
  const spans = groupedSpans[traceId];
  const sortedSpans = spans.sort((a: any, b: any) =>
    (a.startTime?.getTime() || 0) - (b.startTime?.getTime() || 0)
  );

  return {
    traceId,
    spanCount: spans.length,
    startTime: sortedSpans[0]?.startTime?.toISOString() || 'unknown',
    spans: sortedSpans.map((span: any) => ({
      spanId: span.context?.spanId,
      name: span.name,
      attributes: span.attributes || {}
    }))
  };
});

// Build dataset examples using flat top-level fields (required by the AX SDK)
const datasetExamples = traceAnalysis.map(trace => {
  const userQuery = trace.spans[0]?.attributes?.['input.value'] || '';
  const agentResponse = trace.spans[0]?.attributes?.['output.value'] || '';
  const toolCallSpans = trace.spans.filter((span: any) => span.name === 'ai.toolCall');

  return {
    userQuery,
    agentResponse,
    toolCallCount: toolCallSpans.length,
    toolCallData: JSON.stringify(toolCallSpans.map((span: any) => ({
      name: span.name,
      attributes: span.attributes
    }))),
    traceId: trace.traceId,
    source: 'mastra-orchestrator-workflow',
    timestamp: trace.startTime
  };
});

console.log(`Prepared ${datasetExamples.length} examples for dataset`);
console.log('Sample userQuery:', datasetExamples[0]?.userQuery?.slice(0, 100));

const dataset = await createDataset({
  name: `mastra-orchestrator-traces-${Date.now()}`,
  space,
  examples: datasetExamples,
});

console.log('Dataset created:', dataset.id);


# Annotations

Next, we need human annotations to serve as ground truth for evaluation.

Open the dataset in Arize AX and annotate a sample of examples (20–50 is enough to start).
For each example, assess how well the agent's final response aligns with the tool calls and their outputs.

**How to annotate:**
1. Open your dataset in Arize AX
2. Select an example and click **Add Annotation**
3. Set the annotation **name** to exactly `Alignment`
4. Set the **label** to one of: `aligned`, `partially_aligned`, or `misaligned`

**Label definitions:**
- `aligned` — the response is fully supported by the tool outputs
- `partially_aligned` — some information is grounded in tool outputs, but parts are missing or inconsistent
- `misaligned` — the response ignores, contradicts, or invents information not in the tool outputs

> **Stop here.** Annotate your examples in AX before running the cells below.


# LLM Judge Improvement Cycle

Now we'll start with a basic evaluation prompt and improve it iteratively. The workflow looks like this:

**Run the evaluator --> Inspect the outputs and experiment results --> Update the evaluation prompt based on what's lacking --> Repeat until performance improves**

We'll use Arize AX experiments to identify weaknesses in the evaluator, review explanations, and track performance changes over time.

In this notebook, we'll go through two improvement cycles, but you can extend this process with more iterations to fine-tune the evaluator further.

In [11]:
const evalPromptTemplateV1 = `
You are evaluating whether the agent's final response matches the tool outputs.

DATA:
- Query: {{query}}
- Tool Outputs & Response: {{data}}

Choose one label:
- "aligned"
- "partially_aligned"
- "misaligned"

Output only the label.
`;

In [12]:
async function runLlmJudge(
  query: string,
  data: string,
  promptTemplate: string
): Promise<{ label: string }> {
  const prompt = promptTemplate
    .replace("{{query}}", query)
    .replace("{{data}}", data);

  const response = await openai.chat.completions.create({
    model: "gpt-4o",
    messages: [{ role: "user", content: prompt }],
    temperature: 0,
  });

  const label =
    response.choices[0].message.content
      ?.trim()
      .toLowerCase()
      .replace(/['"]/g, "") || "unknown";

  return { label };
}

In [13]:
// Must match the annotation name used in the AX UI
const ANNOTATION_NAME = 'Alignment';

const examples = await listDatasetExamples({ dataset: dataset.id, space });

const annotatedCount = examples.filter(
  (ex: any) => ex.annotations?.some((a: any) => a.name === ANNOTATION_NAME)
).length;

if (annotatedCount === 0) {
  console.warn(`⚠️  No annotations found with name '${ANNOTATION_NAME}'. ` +
    'Annotate examples in AX UI before running this cell. ' +
    'matchScore will be null for all runs.');
}

const experimentRunsV1 = await Promise.all(
  examples.map(async (example) => {
    const ex = example as any;
    const query = ex.userQuery || '';
    const agentData = JSON.stringify({
      agentResponse: ex.agentResponse,
      toolCallData: ex.toolCallData,
    }, null, 2);
    const annotation = ex.annotations?.find(
      (a: any) => a.name === ANNOTATION_NAME
    )?.label;

    const { label } = await runLlmJudge(query, agentData, evalPromptTemplateV1);
    const isMatch = annotation !== undefined && annotation === label;

    console.log({ exampleId: example.id, label, annotation, isMatch });

    return {
      exampleId: example.id,
      output: JSON.stringify({
        evalLabel: label,
        humanAnnotation: annotation ?? 'not_annotated',
        matchesAnnotation: isMatch,
        matchScore: annotation !== undefined ? (isMatch ? 1.0 : 0.0) : null,
      }),
    };
  })
);


In [14]:
const experimentV1 = await createExperiment({
  experimentName: "evalTemplateV1",
  dataset: dataset.id,
  space,
  experimentRuns: experimentRunsV1,
});

console.log("Experiment created:", experimentV1.id);
console.log("View results in the Experiments tab at https://app.arize.com");

Experiment created: RXhwZXJpbWVudDo5NTc3NTptR1M2
View results in the Experiments tab at https://app.arize.com


## Review V1 Results

Use the metrics above to decide where to focus your V2 improvements:

- **Accuracy below 0.70** — re-read the label definitions and tighten them; the prompt likely doesn't make the distinction between labels concrete enough
- **Large skew on one label** — the judge is defaulting to that label; add explicit counter-examples in the prompt for the over-predicted class
- **Low recall on `misaligned`** — add guidance that a response with any fabricated or missing tool output should be `misaligned`, not `partially_aligned`

Next, open the **Experiments** tab in Arize AX and find rows where `matchesAnnotation` is `false`. Read those input/output pairs to understand *why* the judge disagreed, then use those observations to refine the V2 prompt below.

In [ ]:
const parsedRunsV1 = experimentRunsV1.map((r: any) => JSON.parse(r.output));
const annotatedRunsV1 = parsedRunsV1.filter((r: any) => r.humanAnnotation !== 'not_annotated');

if (annotatedRunsV1.length === 0) {
  console.log('⚠️  No annotated examples — annotate in AX UI and re-run cells 16–17 first.');
} else {
  const total = annotatedRunsV1.length;
  const accuracy = annotatedRunsV1.filter((r: any) => r.matchesAnnotation).length / total;

  const humanCounts: Record<string, number> = {};
  const evalCounts: Record<string, number> = {};
  for (const r of annotatedRunsV1) {
    humanCounts[r.humanAnnotation] = (humanCounts[r.humanAnnotation] || 0) + 1;
    evalCounts[r.evalLabel] = (evalCounts[r.evalLabel] || 0) + 1;
  }

  const allLabels = [...new Set([...Object.keys(humanCounts), ...Object.keys(evalCounts)])].sort();

  const perLabel = allLabels.map(label => {
    const tp = annotatedRunsV1.filter((r: any) => r.humanAnnotation === label && r.evalLabel === label).length;
    const fp = annotatedRunsV1.filter((r: any) => r.humanAnnotation !== label && r.evalLabel === label).length;
    const fn = annotatedRunsV1.filter((r: any) => r.humanAnnotation === label && r.evalLabel !== label).length;
    const precision = (tp + fp) > 0 ? tp / (tp + fp) : null;
    const recall = (tp + fn) > 0 ? tp / (tp + fn) : null;
    return { label, precision, recall };
  });

  console.log(`\n📊 V1 Agreement Metrics  (${total} annotated / ${parsedRunsV1.length} total)\n`);
  console.log(`Accuracy: ${(accuracy * 100).toFixed(1)}%\n`);

  console.log('Label distribution (bias check):');
  for (const label of allLabels) {
    const h = humanCounts[label] || 0;
    const e = evalCounts[label] || 0;
    const skewPct = h > 0 ? ((e - h) / h * 100).toFixed(0) : 'n/a';
    console.log(`  ${label.padEnd(20)} human: ${h}   eval: ${e}   skew: ${skewPct}%`);
  }

  console.log('\nPer-label precision & recall:');
  for (const m of perLabel) {
    const p = m.precision !== null ? `${(m.precision * 100).toFixed(1)}%` : 'n/a';
    const r = m.recall !== null ? `${(m.recall * 100).toFixed(1)}%` : 'n/a';
    console.log(`  ${m.label.padEnd(20)} precision: ${p.padEnd(8)} recall: ${r}`);
  }

  // Store accuracy for V1 → V2 comparison in the next section
  (globalThis as any).__metricsV1 = { accuracy };
}

## Review V1 Results

Open the **Experiments** tab in Arize AX and inspect the `evalTemplateV1` experiment.

Look for rows where `matchesAnnotation` is `false` — these are the cases where the LLM judge
disagreed with your human labels. Read through the inputs and outputs for those rows to
understand what the evaluator is getting wrong.

Use those observations to refine the prompt in the next cell before running V2.


In [15]:
const evalPromptTemplateV2 = `
You are evaluating how well an agent's FINAL RESPONSE aligns with the TOOL OUTPUTS it used.

You will be given:
- The original user query
- The agent’s final response
- The tool outputs produced by the agent

QUERY:
{{query}}

TOOL + RESPONSE DATA:
{{data}}

Choose exactly ONE label:

- "aligned" → The final response is fully supported by the tool outputs.
  * Every piece of information in the response can be traced back to the tool calls.
  * There are no additions, fabrications, or contradictions.

- "partially_aligned" → The final response mixes correct tool-based information with extra or inconsistent details.
  * Some information in the response comes from tool outputs, but other parts are missing, fabricated, or inconsistent.
  * The response is only partially grounded in the tool calls.

- "misaligned" → The final response ignores, contradicts, or invents information unrelated to the tool outputs.
  * The tool outputs do not support the response at all, or the response is in direct conflict with them.

Guidelines:
- Focus strictly on whether the content in the final response is supported by the tool outputs.
- Do not reward fluent language or style; only check alignment.
- Provide a short explanation justifying the label.

Your output must contain only one of these labels:
aligned, partially_aligned, or misaligned.
`;

## Compare V1 vs V2

The metrics cell above gives you the V1 → V2 accuracy delta directly.

**Interpreting the comparison:**
- **Accuracy up, skew down** — the prompt refinement worked on both fronts; the judge is more accurate *and* less biased
- **Accuracy up, skew unchanged** — better overall, but the judge still drifts toward one label; tighten the label definitions with counter-examples
- **No improvement** — check that your variable mappings (`userQuery`, `agentResponse`) match what humans were actually reviewing; also check for annotation inconsistency (if annotators disagreed with each other, the evaluator can't converge on a single standard)

Repeat the cycle — inspect mismatches in AX, refine the prompt, create a new experiment — until accuracy exceeds 85%. You can also view the experiments side by side in the Arize AX **Experiments** tab.

In [ ]:
const parsedRunsV2 = experimentRunsV2.map((r: any) => JSON.parse(r.output));
const annotatedRunsV2 = parsedRunsV2.filter((r: any) => r.humanAnnotation !== 'not_annotated');

if (annotatedRunsV2.length === 0) {
  console.log('⚠️  No annotated examples — annotate in AX UI and re-run cells 16–21 first.');
} else {
  const total = annotatedRunsV2.length;
  const accuracy = annotatedRunsV2.filter((r: any) => r.matchesAnnotation).length / total;

  const humanCounts: Record<string, number> = {};
  const evalCounts: Record<string, number> = {};
  for (const r of annotatedRunsV2) {
    humanCounts[r.humanAnnotation] = (humanCounts[r.humanAnnotation] || 0) + 1;
    evalCounts[r.evalLabel] = (evalCounts[r.evalLabel] || 0) + 1;
  }

  const allLabels = [...new Set([...Object.keys(humanCounts), ...Object.keys(evalCounts)])].sort();

  const perLabel = allLabels.map(label => {
    const tp = annotatedRunsV2.filter((r: any) => r.humanAnnotation === label && r.evalLabel === label).length;
    const fp = annotatedRunsV2.filter((r: any) => r.humanAnnotation !== label && r.evalLabel === label).length;
    const fn = annotatedRunsV2.filter((r: any) => r.humanAnnotation === label && r.evalLabel !== label).length;
    const precision = (tp + fp) > 0 ? tp / (tp + fp) : null;
    const recall = (tp + fn) > 0 ? tp / (tp + fn) : null;
    return { label, precision, recall };
  });

  console.log(`\n📊 V2 Agreement Metrics  (${total} annotated / ${parsedRunsV2.length} total)\n`);
  console.log(`Accuracy: ${(accuracy * 100).toFixed(1)}%\n`);

  console.log('Label distribution (bias check):');
  for (const label of allLabels) {
    const h = humanCounts[label] || 0;
    const e = evalCounts[label] || 0;
    const skewPct = h > 0 ? ((e - h) / h * 100).toFixed(0) : 'n/a';
    console.log(`  ${label.padEnd(20)} human: ${h}   eval: ${e}   skew: ${skewPct}%`);
  }

  console.log('\nPer-label precision & recall:');
  for (const m of perLabel) {
    const p = m.precision !== null ? `${(m.precision * 100).toFixed(1)}%` : 'n/a';
    const r = m.recall !== null ? `${(m.recall * 100).toFixed(1)}%` : 'n/a';
    console.log(`  ${m.label.padEnd(20)} precision: ${p.padEnd(8)} recall: ${r}`);
  }

  // V1 → V2 delta (requires the V1 metrics cell to have run first)
  const v1 = (globalThis as any).__metricsV1;
  if (v1) {
    const deltaStr = ((accuracy - v1.accuracy) * 100).toFixed(1);
    const sign = Number(deltaStr) >= 0 ? '+' : '';
    console.log(`\n📈 V1 → V2: ${(v1.accuracy * 100).toFixed(1)}% → ${(accuracy * 100).toFixed(1)}% (${sign}${deltaStr}pp)`);
  }

  if (accuracy >= 0.85) {
    console.log('\n✅ Target threshold reached (≥ 85%). The evaluator is ready to use at scale.');
  } else {
    console.log(`\n⚠️  Below target (${(accuracy * 100).toFixed(1)}% < 85%). Inspect V2 mismatches and iterate the prompt.`);
  }
}

In [16]:
const experimentRunsV2 = await Promise.all(
  examples.map(async (example) => {
    const ex = example as any;
    const query = ex.userQuery || '';
    const agentData = JSON.stringify({
      agentResponse: ex.agentResponse,
      toolCallData: ex.toolCallData,
    }, null, 2);
    const annotation = ex.annotations?.find(
      (a: any) => a.name === ANNOTATION_NAME
    )?.label;

    const { label } = await runLlmJudge(query, agentData, evalPromptTemplateV2);
    const isMatch = annotation !== undefined && annotation === label;

    console.log({ exampleId: example.id, label, annotation, isMatch });

    return {
      exampleId: example.id,
      output: JSON.stringify({
        evalLabel: label,
        humanAnnotation: annotation ?? 'not_annotated',
        matchesAnnotation: isMatch,
        matchScore: annotation !== undefined ? (isMatch ? 1.0 : 0.0) : null,
      }),
    };
  })
);


In [17]:
const experimentV2 = await createExperiment({
  experimentName: "evalTemplateV2",
  dataset: dataset.id,
  space,
  experimentRuns: experimentRunsV2,
});

console.log("Experiment created:", experimentV2.id);
console.log("View results in the Experiments tab at https://app.arize.com");

Experiment created: RXhwZXJpbWVudDo5NTc3NjpzYmNu
View results in the Experiments tab at https://app.arize.com


## Compare V1 vs V2

In the Arize AX **Experiments** tab, compare `evalTemplateV1` and `evalTemplateV2` side by side.

Check the average `matchScore` for each experiment. If V2's score is meaningfully higher,
the refined prompt is better aligned with your human judgment.

Repeat the cycle — inspect mismatches, refine the prompt, create a new experiment — until
your match rate exceeds your target threshold (typically 0.85+).
